# Wimbledon 2026 — Analytical Deep Dive

This notebook answers four analytical questions that complement the prediction model:

1. **SHAP feature analysis** — which features drive predictions, and by how much?
2. **Where does the model fail?** — characterise systematic errors
3. **ELO vs ranking** — does ELO beat raw ATP rankings as a predictor?
4. **Upset rate by round** — does the model's calibration match historical Wimbledon upset patterns?

Run from the repo root with: `jupyter notebook notebooks/analysis.ipynb`

In [ ]:
import sys, os
sys.path.insert(0, '..')   # repo root so tennis_predictor is importable

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from xgboost import XGBClassifier

from tennis_predictor.duckdb_data import load_atp_data, load_atp_rankings, sql
from tennis_predictor.elo import EloEngine
from tennis_predictor.stats import build_stats
from tennis_predictor.features import FEATURE_NAMES, build_features
from tennis_predictor.config import TRAIN_END_YEAR, CALIB_YEAR

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Imports OK')

## Setup: load data and train model
*(~2 minutes — ELO computation over 64k matches)*

In [ ]:
# Load data via DuckDB — replaces 17 CSV read loops with a single SQL scan
df_all   = load_atp_data(start_year=2010, end_year=2026)
df_train = df_all[df_all['tourney_date'].dt.year <= TRAIN_END_YEAR]
df_calib = df_all[df_all['tourney_date'].dt.year == CALIB_YEAR]
df_val   = df_all[(df_all['tourney_date'].dt.year >= 2024) &
                  (df_all['tourney_date'].dt.year <= 2025)]
rankings = load_atp_rankings()

print(f'Train: {len(df_train):,}  Calib: {len(df_calib):,}  Val: {len(df_val):,}')
print(f'Surfaces in training: {df_train["surface"].value_counts().to_dict()}')

In [ ]:
# Build ELO + features
print('Computing ELO ratings...')
engine = EloEngine()
engine.process_dataframe(df_train)
snaps_tr = engine.get_snapshots_df()
stats_tr = build_stats(df_train)
X_tr, y_tr, w_tr = build_features(df_train, snaps_tr, rankings=rankings, **stats_tr)

print(f'Training examples: {len(X_tr):,}  (grass upweighted 3×)')

# Train XGBoost
xgb = XGBClassifier(
    n_estimators=600, max_depth=4, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0, eval_metric='logloss',
    use_label_encoder=False, random_state=42, verbosity=0,
)
xgb.fit(X_tr, y_tr, sample_weight=w_tr)
print('XGBoost trained')

# Calibration
engine.process_dataframe(df_calib)
snaps_cal = engine.get_snapshots_df()
stats_cal = build_stats(pd.concat([df_train, df_calib]))
X_cal, y_cal, _ = build_features(df_calib, snaps_cal, rankings=rankings, **stats_cal)
gi = FEATURE_NAMES.index('is_grass')
gm = X_cal[:, gi] == 1.0
calibrated = CalibratedClassifierCV(xgb, cv='prefit', method='isotonic')
calibrated.fit(X_cal[gm], y_cal[gm])
print('Isotonic calibration fitted')

# Validation set (grass only)
engine.process_dataframe(df_val)
snaps_val = engine.get_snapshots_df()
stats_val = build_stats(pd.concat([df_train, df_calib, df_val]))
X_val, y_val, _ = build_features(df_val, snaps_val, rankings=rankings, **stats_val)
gm_v = X_val[:, gi] == 1.0
X_g, y_g = X_val[gm_v], y_val[gm_v]
print(f'Validation grass samples: {len(X_g):,}')

prob_raw = xgb.predict_proba(X_g)[:, 1]
prob_cal = calibrated.predict_proba(X_g)[:, 1]

---
## Question 1 — SHAP feature analysis

SHAP (SHapley Additive exPlanations) assigns each feature a contribution score **per prediction** rather than a global average.  
This tells us *which* features drive the model's confidence in specific matches — not just overall importance.

In [ ]:
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_g)   # (n_grass_matches, 14)

fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_g, feature_names=FEATURE_NAMES,
    show=False, max_display=14, plot_size=None,
)
plt.title('SHAP values — ATP grass matches (2024–2025 validation)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Mean |SHAP| per feature — clean bar chart
mean_shap = np.abs(shap_values).mean(axis=0)
order     = np.argsort(mean_shap)
names_ord = [FEATURE_NAMES[i] for i in order]
vals_ord  = mean_shap[order]

colors = ['#2980b9' if 'grass' in n or n == 'is_grass' else '#95a5a6' for n in names_ord]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(names_ord, vals_ord, color=colors, edgecolor='white')
ax.set_xlabel('Mean |SHAP value|  (average impact on prediction)', fontsize=11)
ax.set_title('Feature importance (SHAP) — grass-specific features highlighted in blue', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Key insight
top  = names_ord[-1]
top2 = names_ord[-2]
ratio = vals_ord[-1] / vals_ord[FEATURE_NAMES.index('elo_all_diff') if 'elo_all_diff' in [FEATURE_NAMES[i] for i in order] else -3]
print(f'\nInsight: "{top}" is the single most predictive feature (mean |SHAP| = {vals_ord[-1]:.4f}).')
print(f'The top two features together account for {(vals_ord[-1]+vals_ord[-2])/vals_ord.sum()*100:.0f}% of total SHAP mass.')

---
## Question 2 — Where does the model fail?

We look for **systematic** failure patterns by examining matches where the model predicted ≥ 70% confidence for one player but the other won.  
Understanding where models fail is as important as knowing where they succeed.

In [ ]:
errors = (prob_cal >= 0.5).astype(int) != y_g
high_conf_wrong = (prob_cal >= 0.70) & (errors == 1)  # confident & wrong

print(f'Total grass matches:          {len(y_g):,}')
print(f'Total errors (any conf):      {errors.sum():,}  ({errors.mean()*100:.1f}%)')
print(f'High-conf errors (≥70%):      {high_conf_wrong.sum():,}  ({high_conf_wrong.mean()*100:.1f}%)')

In [ ]:
# Error rate by rank gap
rd_idx    = FEATURE_NAMES.index('rank_diff')
rank_diff = X_g[:, rd_idx]

bins   = [-300, -50, -20, -5, 5, 20, 50, 300]
labels = ['< -50', '-50 to -20', '-20 to -5', '≈ 0', '5 to 20', '20 to 50', '> 50']
bin_idx = np.clip(np.digitize(rank_diff, bins) - 1, 0, len(labels) - 1)

error_rates, counts = [], []
for i in range(len(labels)):
    mask = bin_idx == i
    error_rates.append(errors[mask].mean() * 100 if mask.sum() > 0 else 0)
    counts.append(mask.sum())

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(labels, error_rates, color='#c0392b', alpha=0.8, edgecolor='white')
ax.axhline(errors.mean() * 100, color='#2c3e50', linestyle='--', lw=1.5,
           label=f'Overall error rate ({errors.mean()*100:.1f}%)')
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'n={n:,}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Rank difference (positive = model favourite is better ranked)', fontsize=11)
ax.set_ylabel('Error rate (%)', fontsize=11)
ax.set_title('Model error rate by rank gap — ATP grass 2024–2025', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Error rate by ELO diff bucket
elo_idx  = FEATURE_NAMES.index('elo_all_diff')
elo_diff = X_g[:, elo_idx]

ebins   = [-500, -100, -50, -20, 20, 50, 100, 500]
elabels = ['< -100', '-100 to -50', '-50 to -20', '≈ 0', '20 to 50', '50 to 100', '> 100']
ebin_idx = np.clip(np.digitize(elo_diff, ebins) - 1, 0, len(elabels) - 1)

elo_error_rates, elo_counts = [], []
for i in range(len(elabels)):
    mask = ebin_idx == i
    elo_error_rates.append(errors[mask].mean() * 100 if mask.sum() > 0 else 0)
    elo_counts.append(mask.sum())

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(elabels, elo_error_rates, color='#e67e22', alpha=0.85, edgecolor='white')
ax.axhline(errors.mean() * 100, color='#2c3e50', linestyle='--', lw=1.5,
           label=f'Overall error rate ({errors.mean()*100:.1f}%)')
for bar, n in zip(bars, elo_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'n={n:,}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('ELO difference (A − B, positive = A has higher ELO)', fontsize=11)
ax.set_ylabel('Error rate (%)', fontsize=11)
ax.set_title('Model error rate by ELO gap — ATP grass 2024–2025', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nInsight: Error rate is highest when ELO gap is small (≈0) — exactly where')
print('we expect uncertainty. Errors decrease sharply as one player dominates in ELO.')

---
## Question 3 — ELO vs ranking: which predicts better?

We compare three predictors on the 2024–2025 grass holdout:
- **Ranking baseline**: does the higher-ranked player (lower number) win?
- **ELO baseline**: does the player with higher overall ELO win?
- **Full XGBoost model**: all 14 features

In [ ]:
rd_idx   = FEATURE_NAMES.index('rank_diff')
elo_idx  = FEATURE_NAMES.index('elo_all_diff')
gelо_idx = FEATURE_NAMES.index('elo_grass_diff')

# Binary predictions
rank_pred  = (X_g[:, rd_idx]   > 0).astype(int)
elo_pred   = (X_g[:, elo_idx]  > 0).astype(int)
gelo_pred  = (X_g[:, gelо_idx] > 0).astype(int)
model_pred = (prob_cal >= 0.5).astype(int)

# Simple rank-implied probability (for AUC)
rank_prob_raw = X_g[:, rd_idx]  # positive = A is better ranked
# Normalise to [0,1] using a sigmoid-like transform
rank_prob_norm = 1 / (1 + np.exp(-rank_prob_raw / 30))
elo_prob_norm  = 1 / (1 + np.exp(-X_g[:, elo_idx] / 80))

results = {
    'ATP Ranking (binary)':     (accuracy_score(y_g, rank_pred),  None),
    'Overall ELO (binary)':     (accuracy_score(y_g, elo_pred),   None),
    'Grass ELO (binary)':       (accuracy_score(y_g, gelo_pred),  None),
    'ATP Ranking (prob)':        (accuracy_score(y_g, rank_pred),  roc_auc_score(y_g, rank_prob_norm)),
    'Overall ELO (prob)':        (accuracy_score(y_g, elo_pred),   roc_auc_score(y_g, elo_prob_norm)),
    'XGBoost (calibrated)':      (accuracy_score(y_g, model_pred), roc_auc_score(y_g, prob_cal)),
}

print(f'{"Predictor":<30} {"Accuracy":>10} {"AUC-ROC":>10}')
print('-' * 52)
for name, (acc, auc) in results.items():
    auc_str = f'{auc:.4f}' if auc is not None else '  —'
    print(f'{name:<30} {acc:>10.4f} {auc_str:>10}')

In [ ]:
# Visual comparison
labels_v  = ['Ranking\n(prob)', 'Overall ELO\n(prob)', 'XGBoost\n(calibrated)']
acc_vals  = [accuracy_score(y_g, rank_pred),
             accuracy_score(y_g, elo_pred),
             accuracy_score(y_g, model_pred)]
auc_vals  = [roc_auc_score(y_g, rank_prob_norm),
             roc_auc_score(y_g, elo_prob_norm),
             roc_auc_score(y_g, prob_cal)]

x = np.arange(len(labels_v))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, acc_vals, w, label='Accuracy', color='#3498db', alpha=0.85)
b2 = ax.bar(x + w/2, auc_vals, w, label='AUC-ROC',  color='#e74c3c', alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(labels_v, fontsize=11)
ax.set_ylim(0.5, 0.85)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('ELO vs Ranking vs Full Model — ATP grass 2024–2025', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nInsight: Overall ELO outperforms raw ranking by {(roc_auc_score(y_g, elo_prob_norm)-roc_auc_score(y_g, rank_prob_norm))*100:.1f} AUC points.')
print(f'The full XGBoost model adds another {(roc_auc_score(y_g, prob_cal)-roc_auc_score(y_g, elo_prob_norm))*100:.1f} AUC points on top.')

---
## Question 4 — Upset rate by round

Does the model's confidence distribution match the actual historical upset rate at Wimbledon?  
We query the historical data with DuckDB SQL, then overlay the model's predicted probabilities.

In [ ]:
# DuckDB SQL query straight from the query file
upset_query = open('../queries/03_upset_rate_by_round.sql').read()
upset_df = sql(upset_query)
print(upset_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

rounds = upset_df['round'].tolist()
rates  = upset_df['upset_rate_pct'].tolist()
totals = upset_df['total_matches'].tolist()

bars = ax.bar(rounds, rates, color='#16a085', alpha=0.85, edgecolor='white')
for bar, n in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{n} matches', ha='center', va='bottom', fontsize=9, color='#2c3e50')

ax.axhline(25, color='#e74c3c', linestyle='--', lw=1.5, label='25% reference line')
ax.set_xlabel('Round', fontsize=11)
ax.set_ylabel('Upset rate (%)', fontsize=11)
ax.set_title('Historical Wimbledon upset rate by round (2010–2025 ATP)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nInsight: First rounds have a higher upset rate (more mismatches between seeds')
print('and qualifiers). Later rounds are more competitive so upsets become rarer.')

In [ ]:
# Model calibration: predicted win probability distribution vs actual win rate
frac_pos_raw, mean_pred_raw = calibration_curve(y_g, prob_raw, n_bins=10, strategy='quantile')
frac_pos_cal, mean_pred_cal = calibration_curve(y_g, prob_cal, n_bins=10, strategy='quantile')

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(mean_pred_raw, frac_pos_raw, 's--', color='#e67e22', label='Uncalibrated XGBoost', lw=2)
ax.plot(mean_pred_cal, frac_pos_cal, 's-',  color='#2980b9', label='Isotonic calibration', lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')

ax.fill_between([0, 1], [0, 0.95], [0, 1.05], alpha=0.07, color='green', label='±5% band')

ax.set_xlabel('Mean predicted probability', fontsize=12)
ax.set_ylabel('Fraction of positives (actual win rate)', fontsize=12)
ax.set_title('Reliability diagram — ATP grass validation (2024–2025)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

cal_ll = log_loss(y_g, prob_cal)
raw_ll = log_loss(y_g, prob_raw)
print(f'Log-loss improvement from calibration: {raw_ll:.4f} → {cal_ll:.4f}')
print(f'(lower is better — calibration reduced loss by {(raw_ll-cal_ll)*100:.2f} centinats)')

---
## Bonus: Ad-hoc DuckDB queries

The `sql()` helper lets you run any DuckDB query against the raw CSV files.

In [ ]:
# Surface distribution in training data
surface_dist = sql("""
    SELECT surface, COUNT(*) AS matches,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM read_csv_auto('tennis_atp/atp_matches_*.csv', ignore_errors=true)
    WHERE tourney_level IN ('G','M','A','F')
      AND YEAR(STRPTIME(CAST(tourney_date AS VARCHAR), '%Y%m%d')) BETWEEN 2010 AND 2022
    GROUP BY surface
    ORDER BY matches DESC
""")
print('Surface distribution in training data (2010–2022):')
print(surface_dist.to_string(index=False))

In [ ]:
# Top grass win rates (min 20 matches, 2010-2025)
grass_stats = sql(open('../queries/02_grass_stats.sql').read())
print('Top 10 grass win rates (min 20 matches, 2010–2025):')
print(grass_stats.head(10).to_string(index=False))